In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
df = pd.read_csv("../../data/cleaned_cs2_games_pregame.csv")

df.head()

,tournament,team1_id,team1,team2_id,team2,bestOf,map_name,team1_win,team1_player1_id,team1_player2_id,...,team1_player5_id,team2_player1_id,team2_player2_id,team2_player3_id,team2_player4_id,team2_player5_id,year,month,day_of_week,hour
0,10,288898,122,288899,199,5.0,3,0,1401.0,169.0,...,7201.0,19.0,550.0,1549.0,1443.0,146.0,2026,3,6,13
1,10,288898,122,288899,199,5.0,1,0,1401.0,169.0,...,7201.0,19.0,550.0,1549.0,1443.0,146.0,2026,3,6,13
2,10,288894,137,288895,128,3.0,3,1,2988.0,266.0,...,16113.0,1401.0,169.0,1834.0,2934.0,7201.0,2026,3,5,19
3,10,288894,137,288895,128,3.0,4,0,2988.0,266.0,...,16113.0,1401.0,169.0,1834.0,2934.0,7201.0,2026,3,5,19
4,10,288894,137,288895,128,3.0,2,0,2988.0,266.0,...,16113.0,1401.0,169.0,1834.0,2934.0,7201.0,2026,3,5,19


In [ ]:
df.shape

(5569, 22)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5569 entries, 0 to 5568
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   tournament        5569 non-null   int64  
 1   team1_id          5569 non-null   int64  
 2   team1             5569 non-null   int64  
 3   team2_id          5569 non-null   int64  
 4   team2             5569 non-null   int64  
 5   bestOf            5569 non-null   float64
 6   map_name          5569 non-null   int64  
 7   team1_win         5569 non-null   int64  
 8   team1_player1_id  5569 non-null   float64
 9   team1_player2_id  5569 non-null   float64
 10  team1_player3_id  5569 non-null   float64
 11  team1_player4_id  5569 non-null   float64
 12  team1_player5_id  5569 non-null   float64
 13  team2_player1_id  5569 non-null   float64
 14  team2_player2_id  5569 non-null   float64
 15  team2_player3_id  5569 non-null   float64
 16  team2_player4_id  5569 non-null   float64


In [ ]:
target = "team1_win"
y = df[target]
X = df.drop(target, axis=1)

In [ ]:
X = X.drop(columns=["team1_id", "team2_id"])

In [ ]:
## Check for missing values
X.isna().sum()

tournament          0
team1               0
team2               0
bestOf              0
map_name            0
team1_player1_id    0
team1_player2_id    0
team1_player3_id    0
team1_player4_id    0
team1_player5_id    0
team2_player1_id    0
team2_player2_id    0
team2_player3_id    0
team2_player4_id    0
team2_player5_id    0
year                0
month               0
day_of_week         0
hour                0
dtype: int64

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
baseline = y_test.value_counts(normalize=True).max()

print("Baseline Accuracy:", baseline)

Baseline Accuracy: 0.5466786355475763


In [ ]:
## Train simple KNN Classifier
knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test_scaled)

print("KNN Test Accuracy:", accuracy_score(y_test, y_pred))
print("Baseline Accuracy:", baseline)

KNN Test Accuracy: 0.49730700179533216
Baseline Accuracy: 0.5466786355475763


In [ ]:
param_grid = {
    "n_neighbors": list(range(1, 101))
}

grid_search = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=5,
    scoring="accuracy",
    return_train_score=True,
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

print("Best k:", grid_search.best_params_["n_neighbors"])
print("Best validation accuracy:", grid_search.best_score_)

Best k: 1
Best validation accuracy: 0.5472502805836139


In [ ]:
best_knn = grid_search.best_estimator_

y_pred_best = best_knn.predict(X_test_scaled)

print("Best KNN Test Accuracy:", accuracy_score(y_test, y_pred_best))
print("Baseline Accuracy:", baseline)

Best KNN Test Accuracy: 0.5529622980251346
Baseline Accuracy: 0.5466786355475763
